### A smal RAG pipeline

In [1]:
!pip install llama-index

In [2]:
!pip install llama-index-llms-huggingface-api llama-index-embeddings-huggingface

In [3]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from google.colab import userdata
import os

# Retrieve HF_TOKEN from the environment variables
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
hf_token = os.environ["HF_TOKEN"]

llm = HuggingFaceInferenceAPI(
    model_name="Qwen/Qwen2.5-Coder-32B-Instruct",
    temperature=0.7,
    max_tokens=100,
    token=hf_token,
    provider="auto"
)

response = llm.complete("Hello, how are you?")
print(response)
# I am good, how can I help you today?

Hello! I'm just a computer program, so I don't have feelings, but thanks for asking. How can I assist you today?


There are three main ways to load data into LlamaIndex:
1. **SimpleDirectoryReader** : A built-in loader for various file types from a local directory.
2. **LlamaParse** : LlamaParse, LlamaIndex’s official tool for PDF parsing, available as a managed API.
3. **LlamaHub** : A registry of hundreds of data-loading libraries to ingest data from any source.

In [26]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir = "/content/doc")
documents = reader.load_data()
print(documents)

[Document(id_='025c70df-21bf-42e7-b95f-11c5fd0ae0d3', embedding=None, metadata={'file_path': '/content/doc/file1.txt', 'file_name': 'file1.txt', 'file_type': 'text/plain', 'file_size': 767, 'creation_date': '2026-09-29', 'last_modified_date': '2026-09-29'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text="The Kasilof River (/kəˈsiːlɒf/ kə-SEE-lof) or Ggasilatnu in the Dena'ina language is a river on the western Kenai Peninsula in southern Alaska. The name is an anglicization of Reka Kasilova, the name given to the river by early Russian settlers in the area.[1] It begins at Tustumena Lake and flows northwest to Cook Inlet near Kasilof. The 

After loading our documents, we need to break them into smaller pieces called **Node** objects. A **Node** is just a chunk of text from the original document that’s easier for the AI to work with, while it still has references to the original **Document** object.

The **IngestionPipeline** helps us create these nodes through two key transformations.
1. **SentenceSplitter** breaks down documents into manageable chunks by splitting them at natural sentence boundaries.
2. **HuggingFaceEmbedding** converts each chunk into numerical embeddings - vector representations that capture the semantic meaning in a way AI can process efficiently.

In [27]:
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

# Create the pipeline with the transformations
pipeline = IngestionPipeline(
    transformations = [
        SentenceSplitter(chunk_overlap=0),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    ]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

After creating our **Node** objects we need to index them to make them searchable, but before we can do that, we need a place to store our data.

Since we are using an ingestion pipeline, we can directly attach a vector store to the pipeline to populate it. In this case, we will use **Chroma** to store our documents.

In [6]:
!pip install llama-index-vector-stores-chroma

In [28]:

import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path = "./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection = chroma_collection)

pipeline = IngestionPipeline(
    transformations = [
        SentenceSplitter(chunk_size = 25 , chunk_overlap=0),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    ],
    vector_store = vector_store
)


nodes = await pipeline.arun(documents = documents)

print("Documents indexed:", len(nodes))
print("Chroma vectors:", chroma_collection.count())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Metadata length (9) is close to chunk size (25). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Metadata length (9) is close to chunk size (25). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
Documents indexed: 29
Chroma vectors: 29


This is where vector embeddings come in - by embedding both the query and nodes in the same vector space, we can find relevant matches. The **VectorStoreIndex** handles this for us, using the same embedding model we used during ingestion to ensure consistency.

In [29]:
from llama_index.core import VectorStoreIndex

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_vector_store(vector_store , embed_model = embed_model)

# All information is automatically persisted within the ChromaVectorStore object and the passed directory path.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Querying a VectorStoreIndex with prompts and LLMs
Before we can query our index, we need to convert it to a query interface. The most common conversion options are:
- **as_retriever** : For basic document retrieval, returning a list of `NodeWithScore` objects with similarity scores
- **as_query_engine** : For single question-answer interactions, returning a written response
- **as_chat_engine** : For conversational interactions that maintain memory across multiple messages, returning a written response using chat history and indexed context

We’ll focus on the query engine since it is more common for agent-like interactions. We also pass in an LLM to the query engine to use for the response.

In [32]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

llm = HuggingFaceInferenceAPI(
    model_name = "Qwen/Qwen2.5-Coder-32B-Instruct"
)
query_engine = index.as_query_engine(
    llm = llm,
    similarity_top_k = 5,
    # response_mode = "tree_summarize"
    response_mode = "compact"
)
response = query_engine.query("Where is the Kasilof River located?")
print(response)

The Kasilof River is located near an inlet in Alaska, and it is a site where drift and bank fishing for salmon is popular on its lower section. Additionally, there are three Alaska State Parks units situated on or near this river.


### Response Processing
Under the hood, the query engine doesn’t only use the LLM to answer the question but also uses a **ResponseSynthesizer** as a strategy to process the response. Once again, this is fully customisable but there are three main strategies that work well out of the box:
- **refine** : create and refine an answer by sequentially going through each retrieved text chunk. This makes a separate LLM call per Node/retrieved chunk.
- **compact (default)** : similar to refining but concatenating the chunks beforehand, resulting in fewer LLM calls.
- **tree_summarize** : create a detailed answer by going through each retrieved text chunk and creating a tree structure of the answer.

The language model won’t always perform in predictable ways, so we can’t be sure that the answer we get is always correct. We can deal with this by evaluating the quality of the answer.

### Evaluation and observability
LlamaIndex provides built-in evaluation tools to assess response quality. These evaluators leverage LLMs to analyze responses across different dimensions. Let’s look at the three main evaluators available:
- **FaithfulnessEvaluator** : Evaluates the faithfulness of the answer by checking if the answer is supported by the context.
- **AnswerRelevancyEvaluator** : Evaluate the relevance of the answer by checking if the answer is relevant to the question.
- **CorrectnessEvaluator** : Evaluate the correctness of the answer by checking if the answer is correct.

In [39]:
from llama_index.core.evaluation import FaithfulnessEvaluator

query_engine = query_engine
llm = llm

# query index
evaluator = FaithfulnessEvaluator(llm = llm)
response = query_engine.query(
    "What is the Kasilof River?",
    # evaluator = evaluator
)

eval_result = evaluator.evaluate_response(response = response)
eval_result.passing

# print(eval_result.passing)
print(response)

The Kasilof River is a waterway where drift and bank fishing for salmon is popular, particularly in its lower section. It is also associated with the Kasilof River State Recreation Site and is near three Alaska State Parks units.


In [40]:
# install the LlamaTrace callback from Arize Phoenix with the following command:

!pip install -U llama-index-callbacks-arize-phoenix

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 3.7 MB/s eta 0:00:00


In [41]:
# SETUP FOR LlamaTrace -> Use API key that you generated

# import llama_index
# import os

# PHOENIX_API_KEY = "<PHOENIX_API_KEY>"
# os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"api_key={PHOENIX_API_KEY}"
# llama_index.core.set_global_handler(
#     "arize_phoenix",
#     endpoint="https://llamatrace.com/v1/traces"
# )